In [45]:
import pandas as pd

In [46]:
df = pd.read_csv('./csv-folder/food_db.csv')

In [47]:
df.head()

,food_id,nutrient_name,food_class1,food_class2,unit,protein,fiber,vitamin_a,vitamin_c,vitamin_e,...,iron,potassium,magnesium,saturated_fat,added_sugar,sodium,cholesterol,trans_fat,carb,fat
0,D101-004160000-0001,국밥_돼지머리,국밥,돼지머리,900g,6.70,0.7,5.0,0.04,0.0,...,0.24,47.0,5.06,1.47,0.16,181.0,23.82,0.03,15.94,5.16
1,D101-004310000-0001,국밥_순대국밥,국밥,순대국밥,900g,3.17,1.3,5.0,0.21,0.0,...,0.67,34.0,5.80,1.26,0.17,126.0,48.69,0.01,10.38,2.28
2,D101-004500000-0001,국밥_콩나물,국밥,콩나물,780g,1.45,1.2,2.0,1.26,0.0,...,0.18,42.0,5.45,0.12,0.00,172.0,0.00,0.00,10.93,0.24
3,D101-006000000-0001,기장밥,기장밥,해당없음,200g,3.44,1.5,0.0,0.29,0.0,...,0.33,25.0,6.77,0.08,0.00,1.0,0.00,0.00,36.77,0.57
4,D101-007000000-0001,김밥,김밥,해당없음,230g,4.84,1.4,113.0,3.76,0.0,...,0.30,118.0,12.33,1.10,0.00,307.0,19.30,0.02,19.98,4.55


In [48]:
df['food_class1'].unique()

array(['국밥', '기장밥', '김밥', ..., '무 파래무침', '북어채 고추장 무침', '깻잎장아찌'],
      shape=(1234,), dtype=object)

In [49]:
n_df = pd.DataFrame(df['food_class1'].unique(), columns=['food_class1'])
n_df.head()
# n_df.sort_values(by='food_class1', ascending=False, inplace=True)
# n_df.reset_index(drop=True, inplace=True)
# n_df.head()
# n_df.tail()

,food_class1
0,국밥
1,기장밥
2,김밥
3,덮밥
4,보리밥


In [50]:
n_df.to_csv('./csv-folder/food_class1.csv', index=False)

# FOOD_CLASS1.CSV랑 비교하면 됨

In [51]:
df = pd.read_csv('./csv-folder/food_class1.csv')
df.head()

,food_class1
0,국밥
1,기장밥
2,김밥
3,덮밥
4,보리밥


In [69]:
# ===== 세분화된 분류 규칙 =====
# 원본 데이터로 다시 시작
df = pd.read_csv('./csv-folder/food_class1.csv')

# 개별 인식 항목 리스트 (정확히 일치하는 항목만 제외)
exclude_items = [
    '갈비찜', '떡갈비', '오리불고기', '함박스테이크', '비프까스',
    '장어구이', '연어구이', '고등어구이', '갈치구이', '닭발구이', '치킨가스', '햄구이',
    '조기찜', '가자미찜', '가오리찜', '콩나물찜', '김치찜', '오징어찜', '도미찜',
    '제육볶음', '순대볶음', '두부김치', '떡볶이', '고추잡채',
    '감자탕', '김치찌개', '된장찌개', '부대찌개', '순두부찌개', '나베',
    '짬뽕', '메밀소바',
    '육회비빔밥', '육회', '주먹밥', '짬뽕밥',
    '소세지빵', '계란빵', '크로와상', '도넛', '핫케이크', '마카롱', '타르트', '크로플', '베이글', '핫도그', '피자빵', '고로케', '호떡',
    '김치전', '파전', '해물파전',
    '감자튀김', '오징어튀김', '새우튀김',
    '아이스크림', '샤베트', '빙수', '초콜릿', '젤리',
    '수육', '족발', '감바스', '난자완스',
    '시금치나물', '골뱅이무침', '무생채', '닭가슴살샐러드', '닭볶음탕', '고등어조림', '월남쌈'
]

# 분류 함수 정의
def classify_food(df, pattern, category, exclude_list=None):
    """
    음식 분류를 수행하는 헬퍼 함수
    
    Parameters:
    -----------
    df : DataFrame
        분류할 데이터프레임
    pattern : str
        정규식 패턴
    category : str
        분류할 카테고리명
    exclude_list : list, optional
        제외할 항목 리스트 (기본값: None)
    
    Returns:
    --------
    DataFrame
        분류가 적용된 데이터프레임
    """
    if exclude_list:
        mask = df['food_class1'].str.contains(pattern, regex=True) & ~df['food_class1'].isin(exclude_list)
        df.loc[mask, 'food_class1'] = category
    else:
        df.loc[df['food_class1'].str.contains(pattern, regex=True), 'food_class1'] = category
    return df

# 분류 규칙 정의 (패턴, 카테고리명, 제외여부)
# 우선순위: 위에서 아래로 순차적으로 적용되므로, 더 구체적인 패턴을 먼저 배치해야 함
# 예: '회덮밥'은 '회덮밥류'로 먼저 분류되어야 하므로, '덮밥류' 패턴보다 앞에 위치
# 패턴 작성 시 주의사항: 
# - 단일 단어는 ^와 $로 정확히 매칭되도록 작성 (예: '^덮밥$'는 '회덮밥'과 매칭되지 않음)
# - 부분 문자열 매칭을 피하기 위해 정확한 경계를 사용
classification_rules = [
    # 0. 추가 세분화 (가장 구체적인 패턴부터 먼저 처리)
    ('리소토/리조또|^리소토$|^리조또$', '리소토류', False),
    ('조미김구이|김구이', '김구이류', False),
    ('닭발구이|닭발볶음|닭발튀김', '닭발류', True),  # 닭발구이는 개별 분류
    ('소곱창구이|소내장구이|곱창구이|곱창볶음|소곱창볶음|내장전골|순대전골|곱창전골', '소돼지내장류', False),
    ('깐풍새우|깐쇼새우|라조기', '튀김볶음류', False),
    ('잡채$|잡채밥|중식잡채|버섯잡채|어묵잡채|콩나물잡채|돼지고기피망잡채|부추잡채', '잡채류', True),
    ('떡국$|떡만두국', '떡국류', False),
    ('볶음면$|팟타이', '볶음면류', False),
    ('수제비$|바지락수제비', '수제비류', False),
    ('회덮밥$|우럭회덮밥|참치회덮밥', '회덮밥류', False),
    ('국밥$|돼지머리국밥|소고기국밥|콩나물국밥|굴국밥', '국밥류', False),
    ('^자장밥$|^짜장밥$', '짜장밥류', False),
    ('오므라이스|카레라이스|하이라이스', '라이스류', False),
    ('햄버거|버거$', '버거류', False),
    ('스프$|감자스프', '스프류', False),
    ('다시마튀각|김튀김|미역튀각|김부각', '튀각류', False),
    ('강정$|떡강정|닭강정', '강정류', False),
    ('탕수$|탕수육|가지탕수|닭고기탕수|두부탕수|새우탕수|오징어탕수|탕수어', '탕수류', False),
    ('가자미튀김|고등어튀김|꽁치튀김|대구튀김|도미튀김|동태튀김|우럭튀김|임연수튀김|조기튀김|장어튀김|동태까스|미꾸라지튀김|삼치튀김', '생선튀김류', False),
    ('꽃게장|게장$', '게장류', False),
    ('장조림$|돼지고기장조림|메추리알돼지고기장조림|메추리알장조림|소고기장조림|돼지등심장조림', '장조림류', False),
    ('오이무침|오이생채|오이지무침|오이고추된장무침|오이부추무침|오이모듬무침', '오이무침류', False),
    ('굴무침|낙지무침|다시마무침|북어채무침|오징어무침|오징어채무침|톳나물무침|파래무침|피조개무침|해초무침|회무침|꼬막무침|간재미회무침|가오리회무침|문어무침|미역오이무침|뱅어포무침|새우무침|소라무침|쥐포무침|해파리냉채|홍어회무침|오징어미역초무침|갑오징어무침|홍어무침|밴댕이무침|북어채고추장무침', '해산물무침류', False),
    ('오이물김치|열무물김치|나박김치|동치미|물김치$', '물김치류', False),
    ('^콩조림$|^콩자반$|^모듬콩조림$', '콩조림류', False),  # 땅콩조림 제외 (기타조림류에 포함)
    ('꽁치조림|병어조림|삼치조림|양미리조림|쥐포조림|코다리조림|가자미조림|동태조림|북어조림|조기조림|갈치조림|건갈치조림|건꼴뚜기조림|도루묵조림|임연수조림', '생선조림류', True),
    ('감자조림|땅콩조림|무조림|버섯조림|연근조림|우엉조림|고추조림|고구마조림|곤약조림|꽈리고추조림|다시마조림|단호박조림|소시지조림|완자조림|삶은땅콩조림', '기타조림류', False),
    
    # 1. 구이류 세분화
    ('갈비구이|소갈비구이|돼지갈비구이|소갈비찜|돼지갈비찜|돼지등갈비찜|갈비탕|돼지갈비전골|돼지갈비강정', '갈비류', True),
    ('불고기|소불고기|돼지불고기|오리간장불고기|꿩불고기|오징어불고기|불고기전골', '불고기류', True),
    ('스테이크|소등심스테이크|소안심스테이크|돼지고기스테이크', '스테이크류', True),
    ('삼치구이|가자미구이|조기구이|참치구이|임연수구이|전어구이|민어구이|병어구이|적어구이|청어구이|북어구이|넙치구이|도다리구이|동태양념구이|북어양념구이|서대구이|은대구구이|장어양념구이|전갱이구이|조기양념구이|조기오븐구이|삼치양념구이|삼치카레구이|옥돔구이|양태구이|우럭구이|굴비구이|꽁치구이|황태구이', '생선구이류', True),
    ('닭구이|닭날개구이|닭다리구이|닭꼬치구이|치킨데리야끼', '닭구이류', True),
    ('삼겹살구이|돼지껍데기구이|오겹살구이|돼지고기꼬치구이|갈매기살구이', '돼지고기구이류', False),
    ('소곱창구이|소등심구이|소내장구이|차돌박이구이', '소고기구이류', False),
    ('오리고기구이|오리불고기|훈제오리|오리고기로스|오리간장불고기', '오리구이류', True),  # 오리불고기는 개별 분류
    ('새우구이|조개구이|꽃게구이|키조개구이', '해물구이류', False),
    ('버섯구이|감자구이|옥수수구이|더덕구이|채소꼬치구이', '채소구이류', False),
    ('대창구이|막창구이|꼬치구이|꼬치구치|런천미트구이|뱅어포구이|베이컨떡말이구이|칠면조구이|우엉양념구이', '기타구이류', True),
    
    # 2. 찜류 세분화
    ('대구찜|동태찜|북어찜|붕어찜|고등어찜|메기찜|망둥어찜|가오리콩나물찜|대구볼찜|황태콩나물찜', '생선찜류', True),
    ('닭찜', '닭찜류', False),
    ('돼지사태찜|돼지고기찜', '돼지고기찜류', False),
    ('사태찜|소사태찜|소고기찜|소고기떡찜|소고기스튜', '소고기찜류', False),
    ('꼬막찜|해물찜|낙지찜|세발낙지찜|꽃게찜|꽃게콩나물찜|해물콩나물찜|굴찜|바지락찜|명란찜|미더덕찜|복찜|코다리찜', '해물찜류', False),
    ('깻잎찜|꽈리고추찜|애호박찜|풋고추찜|단호박찜|가지찜', '채소찜류', True),
    ('달걀찜|두부.*달걀찜', '달걀찜류', False),
    ('연두부찜', '기타찜류', True),
    
    # 3. 볶음류 세분화
    ('^유산슬$|^팔보채$', '중식해물볶음류', False),
    ('돼지고기볶음|삼겹살볶음|소고기볶음|닭볶음|닭모래집볶음|오리볶음|소간채소볶음|소고기채소볶음|김치돼지고기볶음|낙지삼겹살볶음|주꾸미삼겹살', '고기볶음류', True),
    ('새우볶음|건새우볶음|오징어볶음|오징어채볶음|낙지볶음|문어볶음|주꾸미볶음|세발낙지볶음|^해물볶음$|멸치볶음|잔멸치볶음', '해물볶음류', False),
    ('감자볶음|당근볶음|양파볶음|애호박볶음|호박볶음|호박고지볶음|버섯볶음|브로콜리볶음|마늘쫑볶음|풋고추볶음|꽈리고추볶음|우엉볶음|파래볶음|껍질콩볶음|고추장볶음|가지볶음', '채소볶음류', False),
    ('김치볶음|어묵볶음|두부볶음|라볶이|마파두부|양념두부|폭찹|쫄볶이', '기타볶음류', True),
    
    # 4. 국/탕/찌개/전골 세분화
    ('곰탕|설렁탕|도가니탕|꼬리곰탕|닭곰탕', '곰탕류', False),
    ('매운탕|가자미매운탕|광어매운탕|꽃게매운탕|대구매운탕|대구지리|도미매운탕|메기매운탕|명태매운탕|민어매운탕|복매운탕|복지리|볼락매운탕|붕어매운탕|송어매운탕|숭어매운탕|아귀매운탕|아귀지리|어묵매운탕|우럭매운탕|잉어매운탕|장어매운탕|해물매운탕|동태매운탕|버섯매운탕', '매운탕류', False),
    ('전골|김치전골|낙지전골|버섯전골|소고기전골|불낙전골|두부전골|만두전골|잡탕전골|해물전골|모듬채소전골|국수전골|개고기전골', '전골류', False),
    ('찌개|갈치찌개|꽃게탕|동태찌개|돼지고기찌개|두부찌개|오징어찌개|청국장찌개|추어탕|콩비지찌개|호박찌개|고등어찌개|곱창전골|꽁치찌개|버섯찌개|알탕|도루묵찌개|망둥어찌개|병어찌개|삼치찌개|섞어찌개|조기찌개|참치찌개|콩나물찌개|홍어찌개|고추장찌개|두부명란찌개|두부애호박찌개|명란찌개|애호박찌개|김치순두부|김치청국장찌개|두부청국장찌개|초당순두부|굴두부찌개', '찌개류', True),
    ('국$|개장$|게국지|곰치국|낙지탕|매생이국|묵말이|미소된장국|소탕|아구탕|어탕|연포탕|전복탕|지리탕|홍합탕|황태해장국|무국물|고디국|다슬기맑은국|만두국|달걀탕|달걀국', '국류', False),
    
    # 5. 면류 세분화
    ('국수$|칼국수|콩국수|메밀국수|비빔국수|비빔막국수|잔치국수|쟁반국수|장국국수|해물칼국수|올갱이국수|기스면|쌀국수', '국수류', False),
    ('라면|짬뽕라면', '라면류', False),
    ('우동|우동볶음|볶음우동|어묵우동|중식우동', '우동류', False),
    ('냉면|물냉면|비빔냉면|회냉면', '냉면류', False),
    ('스파게티|미트스파게티|오븐스파게티|치즈스파게티|해물스파게티|나폴리탄스파게티', '스파게티류', False),
    ('^자장$|^짜장$|간자장|쟁반짜장|자장면|짜장면', '자장면류', False),
    ('쫄면|분짜|밀면|탄탄면', '기타면류', True),
    
    # 6. 밥류 세분화
    ('김밥|삼각김밥|충무김밥|모듬김밥', '김밥류', False),
    ('^볶음밥$|김치볶음밥', '볶음밥류', False),
    ('^비빔밥$|비빔잡곡밥|열무비빔밥|낙지비빔밥', '비빔밥류', True),
    ('^덮밥$|돌솥비빔밥|계란덮밥|닭고기덮밥|소고기덮밥|돼지고기덮밥|낙지덮밥|오징어덮밥|장어덮밥|참치덮밥|버섯덮밥|양송이덮밥|해물덮밥', '덮밥류', False),
    ('초밥$|모듬초밥|유부초밥|장어초밥|연어롤|새우튀김롤|캘리포니아롤', '초밥류', False),
    ('굴밥|알밥|영양돌솥밥|찰밥', '기타밥류', True),
    ('밥$|쌀밥|현미밥|흑미밥|순현미밥|잡곡밥|혼합잡곡밥|오곡밥|보리밥|기장밥|수수밥|차조밥|콩밥|팥밥|강낭콩밥|검정콩밥|완두콩밥|옥수수밥|감자밥|고구마밥|밤밥|채소밥|무밥|누룽지|묵밥|연잎밥|곤드레밥|곤드레나물밥|귀리밥|율무밥|잡탕밥', '일반밥류', True),  # 주먹밥, 짬뽕밥, 육회비빔밥은 개별 분류
    
    # 7. 빵류 세분화
    ('빵$|식빵|토스트|소보로빵|크림빵|팥빵|치즈빵|마늘빵|모닝빵|모카빵|버터빵|야채빵|감자빵|건포도빵|기타빵|깨찰빵|롤빵|앙금빵|찰떡빵|허니브레드|호밀빵|효모빵|소금빵|버터크림빵', '일반빵류', True),
    ('머핀|스콘|와플|프레즐|파이|만주|페이스트리|케이크|파운드케이크|카스텔라|다쿠아즈', '베이커리류', True),
    ('샌드위치|트위스터샌드위치', '샌드위치류', True),
    ('피자$', '피자류', True),
    ('또띠아|치아바타|바게트|번|크로켓|츄러스|초코소라빵', '기타빵류', True),
    
    # 8. 떡류
    ('떡$|가래떡|경단|꿀떡|모듬찰떡|백설기|송편|시루떡|약식|인절미|절편|증편|찹쌀떡|개피떡|바람떡|기피편|모싯잎송편|무지개떡|빙떡|메밀전병|수수경단|수수부꾸미|수수팥떡|쑥떡|찰시루떡|떡갈비|떡산적|장떡', '떡류', True),  # 떡갈비, 호떡은 개별 분류
    
    # 9. 죽류
    ('죽$|닭죽|오리죽|전복죽|채소죽|팥죽|호박죽|흑임자죽|게살죽|깨죽|미음|백합죽|소고기버섯죽|쌀죽|흰죽|어죽|율무죽|잣죽|참깨죽|참치죽|현미죽|굴죽|대합죽|땅콩죽|복죽|소고기감자죽|오리감자죽|잡곡죽|홍합미나리죽|홍합소고기죽|들깨죽|애호박죽|찹쌀죽', '죽류', False),
    
    # 10. 전류
    ('전$|가지전|감자전|고구마전|두부부침|맛살전|메밀전|버섯전|부추전|산적|애호박전|완자전|채소전|고추전|굴전|깻잎전|녹두빈대떡|동태전|두부전|미나리전|배추전|새우전|생선전|소고기산적|소라산적|오징어산적|햄부침|호박전|홍합산적|화양적|새송이버섯전|어육소시지전|파래전|녹두전|모듬채소전|소고기완자전|애호박부침개|양파소고기전|어묵전|옥수수부침|해물완자전|햄전|연근전|참치완자전|해물채소전|쑥전|부침개', '전류', True),
    
    # 11. 튀김류
    ('튀김$|돈가스$|닭다리튀김|닭튀김|게맛살튀김|고구마깻잎튀김|깐풍기|깻잎튀김|느타리버섯튀김|달걀튀김|도라지튀김|두부튀김|등심돈가스|소고기튀김|소시지튀김|식빵튀김|쑥튀김|양념돼지고기튀김|어묵튀김|연근튀김|잔멸치튀김|채소튀김|뱅어포튀김|연어튀김|참치강정|고구마맛탕|고구마튀김|고추튀김|김말이튀김|복어튀김|북어강정|생선까스|양파링튀김|쥐포튀김|닭모래집튀김|닭껍데기튀김|북어튀김|우유튀김|감자연근튀김', '튀김류', True),
    
    # 12. 조림류
    ('조림$|닭도리탕|닭조림|두부조림|멸치조림|미트볼조림|어묵조림|오징어조림|오징어채조림|달걀조림|게조림|꽈리고추오징어조림|닭다리조림|무어묵조림|오징어포조림|유부조림', '조림류', True),
    
    # 13. 나물류
    ('나물$|가지나물|고구마줄기나물|고사리나물무침|고춧잎나물무침|깻잎나물|냉이나물|도라지나물무침|머위대나물무침|무나물무침|방풍나물무침|배추나물|비름나물무침|숙주나물|시래기나물무침|씀바귀나물무침|열무나물|유채나물|참나물|취나물|콩나물무침|토란대나물무침|호박오가리나물무침|가죽나물무침|시래기나물|청경채나물|톳나물|고사리나물|고춧잎나물|근대나물|냉이초나물|도라지나물|머위나물|모듬배추나물|무나물|무청나물|방풍나물|부추나물|비름나물|쑥갓나물|씀바귀나물|애호박나물|양배추초나물|얼갈이나물|오이나물|토란대나물|숙주미나리나물|양배추나물|배추숙주나물|호박나물|파강회', '나물류', True),
    
    # 14. 무침류
    ('무침$|겉절이|김무침|냉채$|달래무침|도라지생채|두릅무침|마늘쫑무침|무말랭이무침|묵무침|물회|미나리무침|미역초무침|민들레잎무침|버섯무침|부추무침|양장피|파무침|풋고추된장무침|풋마늘무침|돌나물무침|개고기무침|곤약무침|도토리묵무침|돌나물|마늘장아찌무침|얼갈이겉절이|참나물겉절이|청포묵무침|마늘쫑장아찌무침|메밀묵무침|부추겉절이|부추채소무침|부추초무침|브로콜리두부무침|브로콜리무침|삼색냉채|시금치겉절이|쑥갓오이무침|양념도토리묵|양배추겉절이|양배추부추무침|어묵오이무침|오이도라지생채|유채무침|짜사이무침|참나물무침|천사채무침|치커리무침|콩나물느타리무침|콩나물냉채|노각무침|단무지무침|더덕무침|배추겉절이|상추겉절이|쑥갓나물무침|탕평채|치커리겉절이|미나리초무침|브로콜리채소무침|쑥갓무침|열무겉절이|무생채.*무만|고추장아찌무침|무파래무침|민들레무침|머위나물무침|세발나물무침|우거지나물무침|취나물무침', '무침류', True),
    
    # 15. 생채류
    ('생채|도라지생채|오이생채', '생채류', True),
    
    # 16. 샐러드류
    ('샐러드$|과일샐러드|감자샐러드|단호박샐러드|달걀샐러드|닭고기샐러드|닭고기냉채|닭튀김샐러드|두부냉채|마카로니샐러드|멕시컨샐러드|새싹샐러드|양배추샐러드|양상추샐러드|옥수수샐러드|참치샐러드|햄샐러드|감자샐러드샌드위치|양배추사과샐러드|양배추오이샐러드', '샐러드류', True),
    
    # 17. 김치류
    ('김치$|고들빼기김치|깍두기|깻잎김치|보쌈김치|부추김치|얼갈이배추김치|오이김치|가지김치|섞박지|양념채김치|갓김치|배추김치|백김치|열무김치|열무얼갈이김치|오이소박이|유채김치|유채물김치|총각김치|파김치|양배추김치', '김치류', True),  # 두부김치는 개별 분류
    
    # 18. 젓류
    ('젓$|가자미식해|갈치젓|멸치젓|새우젓|오징어젓|명란젓|조개젓|양념바지락젓|양념새우젓|양념오징어젓', '젓류', False),
    
    # 19. 장아찌류
    ('장아찌$|단무지|더덕장아찌|매실장아찌|무짠지|무초절임|치킨무|오이장아찌|오이지|무말랭이장아찌|무숙장아찌|고추장아찌|들깻잎장아찌|마늘장아찌|마늘쫑장아찌|무장아찌|양파장아찌|깻잎장아찌', '장아찌류', False),
    
    # 20. 수육/족발류
    ('순대$|제육|돼지고기수육|개고기수육', '수육족발류', True),
    
    # 21. 꼬치류
    ('꼬치', '꼬치류', False),
    
    # 22. 만두류
    ('만두$|고기만두|군만두|김치만두|물만두|만두탕수', '만두류', False),
    
    # 23. 음료류
    ('음료$|라떼$|커피$|카페|과.*채주스|귤차|기타음료|기타차|꿀차|녹차|대추차|레몬차|레몬홍차|마테차|밀크티|버블티|뱅쇼|복숭아홍차|사과차|생강차|스무디|식혜|쌍화차|아이스티|에이드|오미자차|우롱차|유자차|자몽차|자몽홍차|코코아|콤부차|탄산음료|허브차|홍차|미숫가루|두유카페라떼|딸기넥타|딸기바나나스무디|미숫가루.*음료|수박화채|아이스카페라떼|아이스카페모카|율무차|카페라떼|카페모카|커피.*타먹는커피|액상커피|레몬청차|밀크쉐이크', '음료류', False),
    
    # 24. 유제품류
    ('요구르트|우유|치즈|두유', '유제품류', False),
    
    # 25. 디저트류
    ('캔디|다식|매작과|산자|약과|유과|팝콘', '디저트류', True),
    
    # 26. 회류
    ('회$|물회|회무침|모듬회', '회류', True),
    
    # 27. 달걀부침류 (기타보다 먼저 적용되어야 함)
    ('^달걀말이$|^달걀부침|^달걀후라이$|^스크램블드에그$|^오믈렛$', '달걀부침류', False),
    
    # 28. 기타
    ('소스|드레싱|카레소스|짜장소스|고춧가루양념장|기름장|쌈장|초간장|양념초장|고구마|옥수수|과일|견과|^달걀$|^계란$|대구포|미숫가루$', '기타', True),
]

# 분류 규칙 적용
for pattern, category, needs_exclude in classification_rules:
    exclude_list = exclude_items if needs_exclude else None
    df = classify_food(df, pattern, category, exclude_list)

print("분류 완료!")
print(f"총 {len(df)}개 항목 분류됨")
print(f"\n고유 분류 수: {df['food_class1'].nunique()}개")
print("\n분류별 개수:")
print(df['food_class1'].value_counts().head(20))

분류 완료!
총 1234개 항목 분류됨

고유 분류 수: 228개

분류별 개수:
food_class1
국류        110
무침류        63
나물류        59
음료류        48
일반밥류       46
전류         44
튀김류        38
찌개류        35
죽류         32
일반빵류       26
해산물무침류     26
매운탕류       25
떡류         25
생선구이류      24
조림류        22
채소볶음류      22
김치류        20
샐러드류       17
장아찌류       16
국수류        15
Name: count, dtype: int64


In [70]:
# 분류 결과 확인 및 저장
print("\n전체 분류 통계:")
print(df['food_class1'].value_counts().sort_index())

# 분류되지 않은 항목 확인 (기타로 분류되지 않은 것들)
unclassified = df[~df['food_class1'].isin(df['food_class1'].value_counts().index)]
if len(unclassified) > 0:
    print(f"\n⚠️ 분류되지 않은 항목: {len(unclassified)}개")
    print(unclassified['food_class1'].unique())


전체 분류 통계:
food_class1
가오리찜    1
가자미찜    1
갈비류     7
갈비찜     1
갈치구이    1
       ..
호떡      1
홍어찜     1
회덮밥류    4
회류      1
훠궈      1
Name: count, Length: 228, dtype: int64


In [71]:
# 분류 결과 저장
df.to_csv('./csv-folder/filtered_food_class.csv', index=False, encoding='utf-8')
print("\n✅ 분류 결과가 'csv-folder/filtered_food_class.csv'에 저장되었습니다.")


✅ 분류 결과가 'csv-folder/filtered_food_class.csv'에 저장되었습니다.
